# Week 4: Logistic Regression and Feature Scaling

In this notebook, I apply logistic regression and feature scaling to the diabetes and Alzheimer's datasets. The goal is to evaluate how well logistic regression performs as a classification model and whether scaling the features improves model performance.

This week focuses on:
- Logistic regression for categorical prediction
- Feature scaling using standardization
- Comparing model performance before and after scaling
- Evaluating classification results using accuracy, confusion matrices, and classification reports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Week 4 Objective

This notebook applies two Week 4 concepts to both project datasets: logistic regression and feature scaling. Logistic regression is used because the target variables represent categorical health outcomes. Feature scaling is tested by comparing logistic regression performance before and after standardizing the input features.

The goal is not only to measure model accuracy, but also to evaluate whether scaling changes the model's performance and whether logistic regression is a reasonable baseline classification model for each dataset.

In [2]:
diabetes_df = pd.read_csv("diabetes_012_health_indicators_BRFSS2015.csv")

diabetes_df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


### Inspecting the Diabetes Dataset

Before building the logistic regression model, I checked the size of the dataset and the distribution of the target variable. This is important because classification models can appear more accurate when one class is much larger than the others.

In [3]:
print("Dataset shape:", diabetes_df.shape)

print("\nTarget distribution:")
print(diabetes_df["Diabetes_012"].value_counts())

print("\nTarget distribution as percentages:")
print(diabetes_df["Diabetes_012"].value_counts(normalize=True) * 100)

Dataset shape: (253680, 22)

Target distribution:
Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

Target distribution as percentages:
Diabetes_012
0.0    84.241170
2.0    13.933302
1.0     1.825528
Name: proportion, dtype: float64


The diabetes target variable is highly imbalanced. Most observations are in class `0`, meaning no diabetes, while class `1`, meaning prediabetes, represents a very small portion of the dataset. Because of this imbalance, accuracy alone may not fully describe model performance. The confusion matrix and classification report will be used to evaluate how well the model performs across all classes.

In [4]:
X_diabetes = diabetes_df.drop("Diabetes_012", axis=1)
y_diabetes = diabetes_df["Diabetes_012"]

print("Feature matrix shape:", X_diabetes.shape)
print("Target vector shape:", y_diabetes.shape)

Feature matrix shape: (253680, 21)
Target vector shape: (253680,)


### Train-Test Split

The diabetes data was split into training and testing sets. I used a stratified split so that the class proportions in the target variable would remain similar in both the training and testing data. This is especially important because the diabetes target variable is imbalanced.

In [5]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.2,
    random_state=42,
    stratify=y_diabetes
)

print("Training feature shape:", X_train_d.shape)
print("Testing feature shape:", X_test_d.shape)

print("\nTraining target distribution:")
print(y_train_d.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test_d.value_counts(normalize=True) * 100)

Training feature shape: (202944, 21)
Testing feature shape: (50736, 21)

Training target distribution:
Diabetes_012
0.0    84.240973
2.0    13.933400
1.0     1.825627
Name: proportion, dtype: float64

Testing target distribution:
Diabetes_012
0.0    84.241958
2.0    13.932908
1.0     1.825134
Name: proportion, dtype: float64


### Logistic Regression Without Feature Scaling

The first model is logistic regression using the original diabetes features. This provides a baseline classification result before applying feature scaling.

In [6]:
log_reg_d = LogisticRegression(max_iter=1000)

log_reg_d.fit(X_train_d, y_train_d)

y_pred_d = log_reg_d.predict(X_test_d)

print("Diabetes Logistic Regression Without Scaling")
print("Accuracy:", accuracy_score(y_test_d, y_pred_d))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_d, y_pred_d))

print("\nClassification Report:")
print(classification_report(y_test_d, y_pred_d))

Diabetes Logistic Regression Without Scaling
Accuracy: 0.8454746136865342

Confusion Matrix:
[[41667     0  1074]
 [  839     0    87]
 [ 5840     0  1229]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.97      0.91     42741
         1.0       0.00      0.00      0.00       926
         2.0       0.51      0.17      0.26      7069

    accuracy                           0.85     50736
   macro avg       0.46      0.38      0.39     50736
weighted avg       0.80      0.85      0.81     50736



/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

The unscaled logistic regression model achieved relatively high accuracy, but the confusion matrix shows that this result is misleading. The model did not predict any observations as class `1`, which represents prediabetes. This caused the precision, recall, and F1-score for class `1` to be 0. The result suggests that the model is heavily influenced by the class imbalance in the diabetes dataset and performs much better on the majority class than on the minority classes.

### Logistic Regression With Feature Scaling

Next, I standardized the diabetes features using `StandardScaler`. Standardization transforms each feature so that it has a mean of 0 and a standard deviation of 1. This helps place the features on a comparable scale before fitting the logistic regression model.

In [7]:
scaler_d = StandardScaler()

X_train_d_scaled = scaler_d.fit_transform(X_train_d)
X_test_d_scaled = scaler_d.transform(X_test_d)

log_reg_d_scaled = LogisticRegression(max_iter=1000)

log_reg_d_scaled.fit(X_train_d_scaled, y_train_d)

y_pred_d_scaled = log_reg_d_scaled.predict(X_test_d_scaled)

print("Diabetes Logistic Regression With Scaling")
print("Accuracy:", accuracy_score(y_test_d, y_pred_d_scaled))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_d, y_pred_d_scaled))

print("\nClassification Report:")
print(classification_report(y_test_d, y_pred_d_scaled))

Diabetes Logistic Regression With Scaling
Accuracy: 0.8454746136865342

Confusion Matrix:
[[41667     0  1074]
 [  839     0    87]
 [ 5840     0  1229]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.97      0.91     42741
         1.0       0.00      0.00      0.00       926
         2.0       0.51      0.17      0.26      7069

    accuracy                           0.85     50736
   macro avg       0.46      0.38      0.39     50736
weighted avg       0.80      0.85      0.81     50736



/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

The scaled logistic regression model produced the same accuracy and confusion matrix as the unscaled model. This suggests that feature scaling did not meaningfully affect performance for the diabetes dataset. One possible reason is that many of the diabetes predictors are already binary or ordinal health indicators, so the feature ranges are not extremely different. However, the scaled model still demonstrates the Week 4 concept of standardization and provides a fair comparison against the unscaled version.

In [8]:
diabetes_results = pd.DataFrame({
    "Dataset": ["Diabetes", "Diabetes"],
    "Model": ["Logistic Regression", "Scaled Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test_d, y_pred_d),
        accuracy_score(y_test_d, y_pred_d_scaled)
    ]
})

diabetes_results

,Dataset,Model,Accuracy
0,Diabetes,Logistic Regression,0.845475
1,Diabetes,Scaled Logistic Regression,0.845475


### Diabetes Model Summary

For the diabetes dataset, logistic regression produced an accuracy of about 84.5% with and without feature scaling. However, the classification report showed that the model failed to predict the prediabetes class. This means the high accuracy was mostly driven by the large number of no-diabetes cases. Feature scaling did not improve performance, likely because many features were already binary or ordinal. Overall, logistic regression provides a useful baseline, but the results show that class imbalance is a major issue for this dataset.

## Alzheimer's Dataset

The second dataset used in this notebook is the Alzheimer's disease dataset. Logistic regression will be used to predict the diagnosis outcome, and model performance will again be compared before and after feature scaling.

In [9]:
alzheimers_df = pd.read_csv("alzheimers_disease_data.csv")

alzheimers_df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,...,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,...,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,...,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,...,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,...,0,0,0.014691,0,0,1,1,0,0,XXXConfid


### Inspecting the Alzheimer's Dataset

Before modeling, I inspected the shape of the Alzheimer's dataset and the distribution of the diagnosis target. The `PatientID` and `DoctorInCharge` columns are not used as predictors because they are identifiers rather than meaningful clinical features.

In [10]:
print("Dataset shape:", alzheimers_df.shape)

print("\nTarget distribution:")
print(alzheimers_df["Diagnosis"].value_counts())

print("\nTarget distribution as percentages:")
print(alzheimers_df["Diagnosis"].value_counts(normalize=True) * 100)

Dataset shape: (2149, 35)

Target distribution:
Diagnosis
0    1389
1     760
Name: count, dtype: int64

Target distribution as percentages:
Diagnosis
0    64.634714
1    35.365286
Name: proportion, dtype: float64


The Alzheimer's diagnosis target is more balanced than the diabetes target. About 64.6% of patients are in class `0`, while about 35.4% are in class `1`. Because the classes are not as uneven as the diabetes dataset, accuracy is somewhat more informative, but the confusion matrix and classification report are still needed to evaluate model performance.

In [11]:
X_alzheimers = alzheimers_df.drop(["Diagnosis", "PatientID", "DoctorInCharge"], axis=1)
y_alzheimers = alzheimers_df["Diagnosis"]

print("Feature matrix shape:", X_alzheimers.shape)
print("Target vector shape:", y_alzheimers.shape)

Feature matrix shape: (2149, 32)
Target vector shape: (2149,)


### Train-Test Split

The Alzheimer's dataset was split into training and testing sets. I again used a stratified split so that the diagnosis class proportions would remain similar in both sets.

In [12]:
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_alzheimers,
    y_alzheimers,
    test_size=0.2,
    random_state=42,
    stratify=y_alzheimers
)

print("Training feature shape:", X_train_a.shape)
print("Testing feature shape:", X_test_a.shape)

print("\nTraining target distribution:")
print(y_train_a.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test_a.value_counts(normalize=True) * 100)

Training feature shape: (1719, 32)
Testing feature shape: (430, 32)

Training target distribution:
Diagnosis
0    64.630599
1    35.369401
Name: proportion, dtype: float64

Testing target distribution:
Diagnosis
0    64.651163
1    35.348837
Name: proportion, dtype: float64


### Logistic Regression Without Feature Scaling

The first Alzheimer's model uses logistic regression on the original unscaled features. This provides a baseline result before applying standardization.

In [13]:
log_reg_a = LogisticRegression(max_iter=1000)

log_reg_a.fit(X_train_a, y_train_a)

y_pred_a = log_reg_a.predict(X_test_a)

print("Alzheimer's Logistic Regression Without Scaling")
print("Accuracy:", accuracy_score(y_test_a, y_pred_a))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_a, y_pred_a))

print("\nClassification Report:")
print(classification_report(y_test_a, y_pred_a))

Alzheimer's Logistic Regression Without Scaling
Accuracy: 0.8162790697674419

Confusion Matrix:
[[243  35]
 [ 44 108]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       278
           1       0.76      0.71      0.73       152

    accuracy                           0.82       430
   macro avg       0.80      0.79      0.80       430
weighted avg       0.81      0.82      0.81       430



/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The unscaled Alzheimer's logistic regression model achieved about 81.6% accuracy. Unlike the diabetes model, this model predicted both classes reasonably well. However, the model produced a convergence warning, meaning the logistic regression solver reached the maximum number of iterations before fully converging. This suggests that feature scaling may help the model train more effectively.

### Logistic Regression With Feature Scaling

Next, I standardized the Alzheimer's features using `StandardScaler`. This is especially useful for this dataset because the features include variables on different scales, such as age, BMI, alcohol consumption, cognitive scores, and symptom indicators.

In [14]:
scaler_a = StandardScaler()

X_train_a_scaled = scaler_a.fit_transform(X_train_a)
X_test_a_scaled = scaler_a.transform(X_test_a)

log_reg_a_scaled = LogisticRegression(max_iter=1000)

log_reg_a_scaled.fit(X_train_a_scaled, y_train_a)

y_pred_a_scaled = log_reg_a_scaled.predict(X_test_a_scaled)

print("Alzheimer's Logistic Regression With Scaling")
print("Accuracy:", accuracy_score(y_test_a, y_pred_a_scaled))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_a, y_pred_a_scaled))

print("\nClassification Report:")
print(classification_report(y_test_a, y_pred_a_scaled))

Alzheimer's Logistic Regression With Scaling
Accuracy: 0.8162790697674419

Confusion Matrix:
[[239  39]
 [ 40 112]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       278
           1       0.74      0.74      0.74       152

    accuracy                           0.82       430
   macro avg       0.80      0.80      0.80       430
weighted avg       0.82      0.82      0.82       430



The scaled Alzheimer's logistic regression model produced the same overall accuracy as the unscaled model, about 81.6%. However, the scaled model slightly improved recall and F1-score for class `1`, which represents diagnosed Alzheimer's cases. The scaled model also avoided the convergence warning that appeared in the unscaled model. This suggests that scaling helped the model train more smoothly, even though it did not increase overall accuracy.

In [15]:
alzheimers_results = pd.DataFrame({
    "Dataset": ["Alzheimer's", "Alzheimer's"],
    "Model": ["Logistic Regression", "Scaled Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_test_a, y_pred_a),
        accuracy_score(y_test_a, y_pred_a_scaled)
    ]
})

alzheimers_results

,Dataset,Model,Accuracy
0,Alzheimer's,Logistic Regression,0.816279
1,Alzheimer's,Scaled Logistic Regression,0.816279


### Alzheimer's Model Summary

For the Alzheimer's dataset, logistic regression achieved about 81.6% accuracy with and without feature scaling. Unlike the diabetes model, the Alzheimer's model predicted both classes reasonably well. Scaling did not improve overall accuracy, but it slightly improved performance for class `1` and removed the convergence warning from the unscaled model. This suggests that feature scaling helped the model train more smoothly.

In [16]:
from sklearn.metrics import f1_score

week4_results = pd.DataFrame({
    "Dataset": ["Diabetes", "Diabetes", "Alzheimer's", "Alzheimer's"],
    "Model": [
        "Logistic Regression",
        "Scaled Logistic Regression",
        "Logistic Regression",
        "Scaled Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test_d, y_pred_d),
        accuracy_score(y_test_d, y_pred_d_scaled),
        accuracy_score(y_test_a, y_pred_a),
        accuracy_score(y_test_a, y_pred_a_scaled)
    ],
    "Macro F1": [
        f1_score(y_test_d, y_pred_d, average="macro"),
        f1_score(y_test_d, y_pred_d_scaled, average="macro"),
        f1_score(y_test_a, y_pred_a, average="macro"),
        f1_score(y_test_a, y_pred_a_scaled, average="macro")
    ]
})

week4_results

,Dataset,Model,Accuracy,Macro F1
0,Diabetes,Logistic Regression,0.845475,0.391581
1,Diabetes,Scaled Logistic Regression,0.845475,0.391581
2,Alzheimer's,Logistic Regression,0.816279,0.796190
3,Alzheimer's,Scaled Logistic Regression,0.816279,0.798721


## Final Week 4 Results

The final results show that logistic regression performed differently across the two datasets. For the diabetes dataset, accuracy was about 84.5%, but the Macro F1-score was only about 0.39. This means the model performed well on the majority class but poorly across all classes overall, especially the prediabetes class.

For the Alzheimer's dataset, accuracy was about 81.6%, while the Macro F1-score was about 0.80. This indicates that the Alzheimer's model performed more evenly across both diagnosis classes. Feature scaling did not improve accuracy for either dataset, but it slightly improved the Macro F1-score for the Alzheimer's model and helped remove the convergence warning.

Overall, logistic regression served as a useful baseline classification model for both datasets. However, the diabetes results show that future models may need to address class imbalance before they can be clinically useful.

## Week 4 Concept Reflection

This notebook applied logistic regression and feature scaling to both project datasets. Logistic regression was appropriate because both target variables represent categorical health outcomes. The diabetes target was multiclass, while the Alzheimer's target was binary.

Feature scaling was tested using standardization. Standardization subtracts the mean and divides by the standard deviation so that features are placed on a comparable scale. This can help logistic regression train more smoothly, especially when features have different units or ranges.

The results showed that scaling did not always improve accuracy. For diabetes, the scaled and unscaled models performed the same. For Alzheimer's, scaling did not improve accuracy, but it slightly improved Macro F1-score and removed the convergence warning. This suggests that scaling can improve model stability even when the final accuracy does not change.

The notebook also showed why classification metrics beyond accuracy are important. The diabetes model had high accuracy but low Macro F1-score because it failed to predict the minority prediabetes class. In healthcare prediction, this matters because a model that ignores smaller patient groups may not be useful in practice.